# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianLogisticRegression, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[ 0.38518708 -0.80188884  0.75037336  0.30109345 -0.93193865]
 [ 0.89262087 -0.82559488  0.17489083  0.16192051  0.18243383]
 [ 0.45447373  0.44208391  0.33917671  0.73181135 -0.92222607]
 [ 0.25650311 -0.51499052 -0.61833323 -0.35362391  0.88035678]
 [ 0.34377215 -0.53313562  0.54480927  0.82437485  0.73241441]
 [-0.19520624 -0.89808462 -0.34068961  0.43272596  0.69985211]
 [-0.68742952  0.64603739 -0.5406911   0.12269711  0.17190558]
 [-0.31872498 -0.67458442 -0.05386499  0.66263795 -0.59662955]
 [-0.31026912 -0.09703329 -0.88330978  0.9943011  -0.40101305]
 [-0.67347741  0.32206931  0.94620081  0.59363865 -0.04499232]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianLogisticRegression(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianLogisticRegression(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a2', 'a2', 'a1', 'a2', 'a1', 'a1', 'a2', 'a1', 'a1', 'a1']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [1, 0, 0, 1, 0, 0, 0, 0, 0, 1]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:34,  1.04s/it]

SVI:   3%|▎         | 1/34 [00:01<00:34,  1.04s/it, loss=761.1512]

SVI:   6%|▌         | 2/34 [00:01<00:33,  1.04s/it, loss=586.2944]

SVI:   9%|▉         | 3/34 [00:01<00:32,  1.04s/it, loss=669.8809]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.15it/s, loss=419.3470]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.15it/s, loss=411.3867]

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.15it/s, loss=447.1209]

SVI: 100%|██████████| 34/34 [00:01<00:00, 23.75it/s, loss=447.1209]

SVI: 100%|██████████| 34/34 [00:01<00:00, 23.75it/s, loss=423.9324]